In [1]:
import openpyxl
import pandas as pd
from openpyxl.utils import get_column_letter
import re

# Load the workbook
wb = openpyxl.load_workbook('PRISMA_Deduplicated.xlsx')
ws = wb['Duplicate Review(DOI + Title)']

# Read data into a list of dictionaries, SKIP EMPTY ROWS
headers = []
for col in range(1, ws.max_column + 1):
    headers.append(ws.cell(1, col).value)

data = []
for row in range(2, ws.max_row + 1):
    record = {}
    is_empty = True
    for col_idx, header in enumerate(headers, start=1):
        value = ws.cell(row, col_idx).value
        record[header] = value
        if value is not None:
            is_empty = False
    
    if not is_empty:
        data.append(record)

print(f"Total records before keyword filtering: {len(data)}")

# Convert to DataFrame
df = pd.DataFrame(data)

# Create full text column (combining Title, Abstract, Keywords)
df['Full_Text'] = (
    df['Title'].fillna('').astype(str) + ' ' + 
    df['Abstract'].fillna('').astype(str) + ' ' + 
    df['Keywords'].fillna('').astype(str)
)

# Convert to lowercase for case-insensitive matching
df['Full_Text_Lower'] = df['Full_Text'].str.lower()

# Define keyword sets for each category
open_source_keywords = [
    'open source', 'open-source', 'opensource',
    'foss', 'free software', 'open data',
    'public domain', 'creative commons',
    'open access', 'open-access',
    'permissive license', 'mit license', 'apache license',
    'gpl', 'gnu', 'copyleft',
    'freely available', 'publicly available',
    'open platform', 'open standard', 'open protocol',
    'open code', 'shared resource', 'community-driven',
    'collaborative development', 'transparent', 'transparency'
]

ai_ml_keywords = [
    'artificial intelligence', 'machine learning', 'deep learning',
    'neural network', 'ai', 'ml', 'dl', 'ann',
    'natural language processing', 'nlp',
    'computer vision', 'reinforcement learning',
    'supervised learning', 'unsupervised learning',
    'algorithm', 'model training', 'predictive model',
    'data mining', 'pattern recognition',
    'cognitive computing', 'intelligent system',
    'automated learning', 'statistical learning',
    'convolutional', 'recurrent neural', 'lstm', 'gru',
    'random forest', 'decision tree', 'support vector',
    'svm', 'knn', 'k-nearest', 'naive bayes',
    'clustering', 'classification', 'regression',
    'ensemble learning', 'boosting', 'bagging',
    'neural', 'network', 'learning algorithm',
    'predictive analytics', 'intelligent', 'smart',
    'adaptive', 'autonomous', 'automated',
    'computational intelligence', 'soft computing',
    'expert system', 'knowledge-based',
    'optimization', 'heuristic', 'metaheuristic',
    'genetic algorithm', 'evolutionary',
    'fuzzy logic', 'fuzzy system',
    'data-driven', 'model-based',
    'prediction', 'forecasting', 'inference',
    'training', 'testing', 'validation'
]

security_governance_keywords = [
    'security', 'cybersecurity', 'cyber security', 'cyber-security',
    'governance', 'compliance', 'regulation', 'regulatory',
    'privacy', 'data protection', 'gdpr', 'privacy-preserving',
    'encryption', 'authentication', 'authorization',
    'access control', 'risk management', 'audit', 'auditing',
    'vulnerability', 'threat', 'attack', 'intrusion',
    'secure', 'safety', 'trust', 'trustworthy', 'trusted',
    'policy', 'framework', 'standard', 'guideline',
    'iso', 'nist', 'regulatory framework',
    'data governance', 'information security', 'infosec',
    'risk assessment', 'security framework',
    'data privacy', 'confidentiality', 'integrity', 'availability',
    'cia triad', 'security control', 'security measure',
    'threat model', 'risk', 'secure communication',
    'cryptography', 'cryptographic', 'digital signature',
    'identity management', 'access management',
    'permission', 'role-based', 'rbac',
    'malicious', 'adversarial', 'attack vector',
    'defense', 'protection', 'safeguard',
    'monitoring', 'detection', 'prevention',
    'incident response', 'breach', 'compromise',
    'secure design', 'security architecture',
    'best practice', 'security policy',
    'accountability', 'traceability', 'auditability',
    'verification', 'validation', 'assurance',
    'resilience', 'robustness', 'reliability',
    'safety-critical', 'mission-critical',
    'ethical', 'ethics', 'responsible', 'fairness',
    'bias', 'explainability', 'interpretability',
    'accountability framework', 'oversight'
]

# Function to check if any keyword from a list is present in text
def contains_keyword(text, keyword_list):
    if pd.isna(text) or text == '':
        return False
    text_lower = text.lower()
    for keyword in keyword_list:
        # Use word boundaries for more accurate matching
        if re.search(r'\b' + re.escape(keyword) + r'\b', text_lower):
            return True
    return False

# Apply filters (AND logic - all three conditions must be true)
df['Has_OpenSource'] = df['Full_Text_Lower'].apply(lambda x: contains_keyword(x, open_source_keywords))
df['Has_AI_ML'] = df['Full_Text_Lower'].apply(lambda x: contains_keyword(x, ai_ml_keywords))
df['Has_Security_Governance'] = df['Full_Text_Lower'].apply(lambda x: contains_keyword(x, security_governance_keywords))

# Filter records that match ALL three criteria
df_filtered = df[df['Has_OpenSource'] & df['Has_AI_ML'] & df['Has_Security_Governance']].copy()

print(f"\nFiltering results:")
print(f"Records with Open Source keywords: {df['Has_OpenSource'].sum()}")
print(f"Records with AI/ML keywords: {df['Has_AI_ML'].sum()}")
print(f"Records with Security/Governance keywords: {df['Has_Security_Governance'].sum()}")
print(f"Records matching ALL criteria (AND): {len(df_filtered)}")

# Drop the helper columns before saving
df_filtered = df_filtered[headers]

# Create new sheet for filtered records
sheet_name = 'Duplicate Review(DOI + Title + KW)'
if sheet_name in wb.sheetnames:
    del wb[sheet_name]
ws_new = wb.create_sheet(sheet_name)

# Write headers
for col_idx, header in enumerate(headers, start=1):
    ws_new.cell(1, col_idx, header)
    ws_new.cell(1, col_idx).font = openpyxl.styles.Font(bold=True)

# Write filtered data
for row_idx, record in enumerate(df_filtered.to_dict('records'), start=2):
    for col_idx, header in enumerate(headers, start=1):
        ws_new.cell(row_idx, col_idx, record[header])

# Auto-adjust column widths
for col_idx, header in enumerate(headers, start=1):
    column_letter = get_column_letter(col_idx)
    if header == 'Title':
        ws_new.column_dimensions[column_letter].width = 50
    elif header == 'Author/(s)':
        ws_new.column_dimensions[column_letter].width = 30
    elif header == 'Abstract':
        ws_new.column_dimensions[column_letter].width = 60
    elif header == 'Keywords':
        ws_new.column_dimensions[column_letter].width = 30
    elif header == 'Year':
        ws_new.column_dimensions[column_letter].width = 10
    elif header == 'Platform':
        ws_new.column_dimensions[column_letter].width = 20
    elif header == 'DOI':
        ws_new.column_dimensions[column_letter].width = 25

# Save the workbook
wb.save('PRISMA_Deduplicated.xlsx')

print(f"\nKeyword filtering complete!")
print(f"Filtered records ({len(df_filtered)}) saved to '{sheet_name}' sheet")
print(f"Original 'Duplicate Review(DOI + Title)' sheet remains untouched")

/opt/conda/envs/anaconda-2024.02-py310/lib/python3.10/site-packages/openpyxl/workbook/child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")


Total records before keyword filtering: 6279

Filtering results:
Records with Open Source keywords: 1016
Records with AI/ML keywords: 5664
Records with Security/Governance keywords: 5925
Records matching ALL criteria (AND): 749


/opt/conda/envs/anaconda-2024.02-py310/lib/python3.10/site-packages/openpyxl/workbook/child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")



Keyword filtering complete!
Filtered records (749) saved to 'Duplicate Review(DOI + Title + KW)' sheet
Original 'Duplicate Review(DOI + Title)' sheet remains untouched
